# Supervised attack classification: the original design

The reservoir grows (220 -> 10,626 Fock states) while the classical head stays a
**single linear layer** (full-batch L-BFGS softmax). This is the design whose
results flattened out, reproduced by the shared runner so it can be compared
line for line with `11_supervised_scaled_head`.

Arms: `quantum` (the reservoir readout), `dials` (the PCA input that drove it,
the tight control) and `raw` (all features, the classical ceiling).
Config: `configs/supervised_reference.yaml`. Historical results of the original
notebook are in `runs/legacy/qorc_sup_runs/`.

In [ ]:
# Setup: make the repo importable from notebooks/ (or from the repo root).
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
sys.path.insert(0, str(REPO))

import pandas as pd
from elements import config
from elements.eval.results import load_results
from elements.experiments.runner import plan, run
from elements.viz import scaling
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
from elements.viz import supervised as sv

In [ ]:
CFG = config.load(REPO / "configs" / "supervised_reference.yaml")
FIGS = CFG.out_dir / "figs"
print(plan(CFG))

### Run

Off by default so opening the notebook never starts a long run. Set `RUN = True`
(or run `python -m elements.run configs/supervised_reference.yaml` from the repo root). Every trained head is
appended to `results.jsonl` as it finishes, so an interrupted run resumes where
it stopped.

In [ ]:
RUN = False
if RUN:
    run(CFG)

## Results

In [ ]:
DF = load_results(CFG.out_dir, keep=("per_class_f1", "confusion"))
scaling.capacity(DF, "macro_f1", head="lbfgs", arms=("quantum", "dials"),
                 out=FIGS / "capacity.png");

If the quantum line rises while `dials` stays flat, the reservoir contributes.
If the quantum line flattens below the dashed raw-input line, the linear head
has run out of room -- the question `11_supervised_scaled_head` answers.

In [ ]:
scaling.gain_heatmap(DF, "macro_f1", "dials", out=FIGS / "gain_vs_dials.png");

### Per-class F1 gain and confusion

In [ ]:
sv.per_class_gain(DF, "all23", head="lbfgs", out=FIGS / "per_class_gain_all23.png");

In [ ]:
LAST = scaling.config_order(DF)[-1]
sv.confusion(DF, "families6", LAST, "lbfgs", out=FIGS / f"confusion_{LAST}.png");

### Operational view

`false_alarm_rate`: normal traffic flagged as an attack. `miss_rate`: attacks
passed as normal. `attack_type_acc`: of attacks caught as attacks, how often the
type is right.

In [ ]:
sv.operational_table(DF)

**Caveats.** One seed: differences under ~0.01-0.02 (and ~0.05 on `all23`, whose
tail classes have 2-11 rows) are not established. This is closed-set
classification of known attacks; zero-day detection is `20_*` / `21_*`.